# WooCommerce Supplier Repor products -- Connecting to Your Database

This notebook connects directly to your WordPress/WooCommerce MySQL database, pulls
sales data into a pandas DataFrame, and lets you analyze it in Python -- automating
the same reconciliation the data for create report.

**What you'll learn in this notebook:**
1. Connecting Python to a remote MySQL database
2. Handling credentials safely (never hardcode passwords!)
3. Running SQL queries and loading results into pandas
4. Exploring a DataFrame
5. A basic chart
6. Exporting results

## Step 1 -- Install the libraries we need
- **pymysql** -- lets Python talk to MySQL (pure Python, nothing extra to compile)
- **sqlalchemy** -- gives pandas a standard, clean way to connect to a database
- **pandas** -- turns SQL results into a DataFrame you can filter, group, and plot
- **python-dotenv** -- loads secret credentials from a separate file, so they never
  end up written inside this notebook
- **matplotlib** -- for the chart in Step 8

In [115]:
#This is the command
!pip install pymysql sqlalchemy pandas python-dotenv matplotlib


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [116]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

## Step 2 -- Set up your credentials safely
**Never write your database password directly inside a notebook.** If you ever share
this file, push it to GitHub, or someone looks over your shoulder, that password is
exposed in plain text.

Instead, create a plain text file named `.env` in the **same folder** as this
notebook, with content like this (use your real values):

```
DB_HOST=auth-db1839.hstgr.io
DB_PORT=3306
DB_NAME=your_database_name
DB_USER=your_database_user
DB_PASSWORD=your_database_password
```

If this ever becomes a git project, add a `.gitignore` file containing just `.env`
so it never gets committed by accident.

## 2. Connect to the database

Credentials come from a local `.env` file, never hardcoded here -- see
`.env.example` next to this notebook. Copy it to `.env`, fill in your real
values, and keep `.env` out of version control (add it to `.gitignore`).

If you're running this notebook from your own machine rather than on the
server itself, you'll likely need to enable **Remote MySQL** access for this
database in Hostinger's hPanel and allow your current IP -- shared hosting
only accepts local connections by default.

In [117]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    print("Connected OK")

Connected OK


In [118]:
## 2. Set the period to check

START_DATE = "2026-08-01"   # inclusive
END_DATE   = "2026-09-01"   # exclusive

#Change these two dates and every query below updates automatically.

query_orders = f"""
SELECT
    p.ID AS order_id,
    p.post_status AS status,
    p.post_date_gmt AS date_created_gmt,
    MAX(CASE WHEN pm.meta_key = '_order_total'    THEN pm.meta_value END) + 0 AS order_total,
    MAX(CASE WHEN pm.meta_key = '_order_tax'      THEN pm.meta_value END) + 0 AS order_tax,
    MAX(CASE WHEN pm.meta_key = '_order_shipping' THEN pm.meta_value END) + 0 AS order_shipping,
    (
        MAX(CASE WHEN pm.meta_key = '_order_total'    THEN pm.meta_value END) -
        MAX(CASE WHEN pm.meta_key = '_order_tax'      THEN pm.meta_value END) -
        MAX(CASE WHEN pm.meta_key = '_order_shipping' THEN pm.meta_value END)
    ) + 0 AS net_sales_recalculated
FROM wp_posts p
JOIN wp_postmeta pm ON pm.post_id = p.ID
WHERE p.post_type = 'shop_order'
  AND p.post_status NOT IN ('wc-pending', 'wc-cancelled', 'wc-failed', 'trash')
  AND p.post_date_gmt BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY p.ID
"""

df_orders = pd.read_sql(query_orders, engine)
df_orders

,order_id,status,date_created_gmt,order_total,order_tax,order_shipping,net_sales_recalculated
0,4913,wc-completed,2026-08-08 14:55:31,"240,000",0,0,"240,000"
1,4923,wc-completed,2026-08-03 15:53:49,"43,000",0,0,"43,000"
2,4924,wc-completed,2026-08-03 17:09:25,"329,300",0,0,"329,300"
3,4925,wc-completed,2026-08-03 19:01:37,"96,000",0,0,"96,000"
4,4927,wc-completed,2026-08-04 23:50:11,"170,600",0,0,"170,600"
5,4929,wc-completed,2026-08-05 19:29:36,"15,504",0,0,"15,504"
6,4931,wc-completed,2026-08-05 19:33:10,"136,500",0,0,"136,500"
7,4932,wc-completed,2026-08-05 19:42:25,"72,000",0,0,"72,000"
8,4936,wc-completed,2026-08-07 01:01:23,"240,400",0,0,"240,400"
9,4937,wc-completed,2026-08-08 21:58:01,"77,297",0,0,"77,297"


In [119]:
# print(df_orders["order_total"])
df_orders_total = df_orders["order_total"].sum()
print(df_orders_total)

8841993.0


In [123]:
START_DATE = "2026-08-01 00:00:00"
END_DATE   = "2026-08-31 23:59:59"

EXCLUDED_SQL = ", ".join(f"'{s}'" for s in EXCLUDED)

query_sku = f"""

SELECT
    li.order_id,
    li.status,
    li.order_item_id,
    li.product_id,
    li.variation_id,
    COALESCE(NULLIF(vsku.meta_value, ''), psku.meta_value) AS sku,
    li.product_name,
    li.qty,
    li.line_subtotal,
    li.line_total,
    li.line_tax,
    (li.line_subtotal - li.line_total) AS discount
FROM (
    SELECT
        oi.order_item_id,
        oi.order_id,
        o.post_status      AS status,
        oi.order_item_name AS product_name,
        CAST(MAX(CASE WHEN oim.meta_key='_product_id'   THEN oim.meta_value END) AS UNSIGNED) AS product_id,
        CAST(MAX(CASE WHEN oim.meta_key='_variation_id' THEN oim.meta_value END) AS UNSIGNED) AS variation_id,
        COALESCE(MAX(CASE WHEN oim.meta_key='_qty'           THEN oim.meta_value END), 0) + 0 AS qty,
        COALESCE(MAX(CASE WHEN oim.meta_key='_line_subtotal' THEN oim.meta_value END), 0) + 0 AS line_subtotal,
        COALESCE(MAX(CASE WHEN oim.meta_key='_line_total'    THEN oim.meta_value END), 0) + 0 AS line_total,
        COALESCE(MAX(CASE WHEN oim.meta_key='_line_tax'      THEN oim.meta_value END), 0) + 0 AS line_tax
    FROM wp_woocommerce_order_items oi
    JOIN wp_woocommerce_order_itemmeta oim
      ON oim.order_item_id = oi.order_item_id
     AND oim.meta_key IN ('_product_id','_variation_id','_qty','_line_subtotal','_line_total','_line_tax')
    JOIN wp_posts o ON o.ID = oi.order_id
    WHERE oi.order_item_type = 'line_item'
      AND o.post_type    = 'shop_order'
      AND o.post_status  = 'wc-completed'
      AND o.post_date   >= '{START_DATE}'
      AND o.post_date   <= '{END_DATE}'
    GROUP BY oi.order_item_id, oi.order_id, o.post_status, oi.order_item_name
) li
LEFT JOIN wp_postmeta psku ON psku.post_id = li.product_id   AND psku.meta_key = '_sku'
LEFT JOIN wp_postmeta vsku ON vsku.post_id = li.variation_id AND vsku.meta_key = '_sku'
ORDER BY li.order_id, li.order_item_id

"""

df_sku = pd.read_sql(query_sku, engine)


In [121]:
df_sku_total = df_sku["line_total"].sum()
print(f"Subtotal: {df_sku_total:,.2f}")

Subtotal: 8,443,601.00


In [129]:
!pip install gspread gspread-dataframe google-auth


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os, requests, pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

url   = os.getenv("APPS_SCRIPT_URL")
token = os.getenv("APPS_SCRIPT_TOKEN")

if not url or not token:
    raise RuntimeError("APPS_SCRIPT_URL or APPS_SCRIPT_TOKEN missing from .env")

resp = requests.get(url, params={"token": token, "sheet": "PRECIO"}, timeout=30)
resp.raise_for_status()

data = resp.json()
if isinstance(data, dict):
    if "error" in data:
        raise RuntimeError(data["error"])
    data = data.get("rows", data)

df_precios = pd.DataFrame(data)

# The SKU column has no header in the sheet, so it arrives as ""
df_precios.columns = [str(c).strip() for c in df_precios.columns]
df_precios = df_precios.rename(columns={"": "sku"})

PRICE_COLS = ["COSTO", "PRECIO MAYORISTA", "TECNICO", "PRECIO FINAL SIN IVA"]

# "PAPELERA" and other text markers become NaN
for c in PRICE_COLS:
    df_precios[c] = pd.to_numeric(df_precios[c], errors="coerce")

df_precios["sku"] = df_precios["sku"].astype(str).str.strip().str.upper()
df_precios["PRODUCTOS"] = df_precios["PRODUCTOS"].astype(str).str.strip()

df_precios = df_precios[df_precios["sku"].ne("") & df_precios["sku"].ne("NAN")]

print(f"{len(df_precios)} rows")
print("Sin precio:", df_precios["COSTO"].isna().sum())
print("SKU duplicados:", df_precios["sku"].duplicated().sum())

df_precios.info()
df_precios.head()